# DS201 - ĐỒ ÁN HỌC SÂU TRONG KHOA HỌC DỮ LIỆU
## Notebook 03: Khám phá Toàn diện Dữ liệu Số (Numerical) & Nhãn Mục tiêu (isFraud)

**Mục tiêu của Notebook này:**
1. **Nhãn `isFraud`:** Thống kê chính xác tỷ lệ gian lận nền và phân tích trôi dạt khái niệm theo thời gian (Concept Drift qua từng tuần/tháng).
2. **Số tiền `TransactionAmt`:** Phân tích phân vị, các giao dịch cực đoan (outliers), và kiểm chứng giả thuyết về phần thập phân số tiền lẻ (`cents`).
3. **Nhóm biến đếm `C1 - C14`:** Khám phá độ lệch phân phối và các giá trị đếm bất thường.
4. **Nhóm khoảng cách thời gian `D1 - D15`:** Đánh giá phân phối ngày thực tế và các giá trị âm (lệch giờ).
5. **Giải mã 339 cột `V1 - V339`:** Gom nhóm 339 cột theo mẫu hình missing thành 15 cụm mẹ.
6. **Soi sâu 15 Cụm Mẹ của cột V:** Phân tích bản chất nội dung (Binary, Count, Amount), đo độ trùng lặp tương quan và chọn lọc đại diện tinh hoa.

### Khối 0: Cấu hình Môi trường & Nạp Dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

if os.path.exists(os.path.join("data", "raw", "ieee-fraud-detection")):
    DATA_DIR = os.path.join("data", "raw", "ieee-fraud-detection")
elif os.path.exists(os.path.join("..", "data", "raw", "ieee-fraud-detection")):
    DATA_DIR = os.path.join("..", "data", "raw", "ieee-fraud-detection")
else:
    raise FileNotFoundError("Không tìm thấy thư mục data/raw/ieee-fraud-detection/")

PATH_TRANS = os.path.join(DATA_DIR, "train_transaction.csv")
PATH_IDENT = os.path.join(DATA_DIR, "train_identity.csv")

print(f"[*] Đường dẫn dữ liệu xác định tại: {DATA_DIR}")
print("[*] Đang nạp toàn bộ dữ liệu vào RAM (chỉ thực hiện 1 lần duy nhất)... ")

df_trans = pd.read_csv(PATH_TRANS)
df_ident = pd.read_csv(PATH_IDENT)
df = pd.merge(df_trans, df_ident, on='TransactionID', how='left')

del df_trans, df_ident
gc.collect()

print(f"=> Nạp thành công! Bảng dữ liệu có: {df.shape[0]:,} dòng và {df.shape[1]} cột.")

---
### Khối 1: Sự thật về Nhãn `isFraud` & Tính Ổn định theo Thời gian (Concept Drift)

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100

print("=" * 60)
print("THỐNG KÊ TỔNG THỂ NHÃN MỤC TIÊU (isFraud):")
print(f" - Giao dịch HỢP LỆ (0):  {fraud_counts[0]:>8,} dòng ({fraud_pct[0]:.3f}%)")
print(f" - Giao dịch GIAN LẬN (1): {fraud_counts[1]:>8,} dòng ({fraud_pct[1]:.3f}%)")
print(f" => Tỷ số mất cân bằng: 1 vụ gian lận trên mỗi {fraud_counts[0]/fraud_counts[1]:.1f} giao dịch bình thường.")
print("=" * 60)

df['week'] = (df['TransactionDT'] // (7 * 86400)).astype(int)

weekly_stats = df.groupby('week')['isFraud'].agg(total_tx='count', fraud_tx='sum', fraud_rate='mean').reset_index()
weekly_stats['fraud_rate_pct'] = weekly_stats['fraud_rate'] * 100

print("\n[Top 5 tuần đầu và 5 tuần cuối]:")
display(pd.concat([weekly_stats.head(5), weekly_stats.tail(5)]))

plt.figure(figsize=(12, 4))
plt.plot(weekly_stats['week'], weekly_stats['fraud_rate_pct'], marker='o', color='crimson', linewidth=2)
plt.axhline(fraud_pct[1], color='gray', linestyle='--', label=f'Trung bình toàn bảng ({fraud_pct[1]:.2f}%)')
plt.title('Tỷ lệ Gian lận (%) Biến thiên theo Tuần trong 6 Tháng (Concept Drift)', fontsize=13, fontweight='bold')
plt.xlabel('Tuần thứ (0 đến 25)', fontsize=11)
plt.ylabel('Tỷ lệ Gian lận (%)', fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

---
### Khối 2: Phân tích Số tiền (`TransactionAmt`) & Phần thập phân (`cents`)

In [ ]:
amt_percentiles = df['TransactionAmt'].quantile([0, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99, 0.999, 1.0])
print("=" * 60)
print("PHÂN VỊ SỐ TIỀN GIAO DỊCH (TransactionAmt tính bằng USD):")
for q, val in amt_percentiles.items():
    print(f" - Phân vị {q*100:5.1f}%: ${val:>10.2f}")
print("=" * 60)

df['amt_cents'] = np.round(df['TransactionAmt'] - np.floor(df['TransactionAmt']), 4)

has_cents_0 = (df[df['isFraud'] == 0]['amt_cents'] > 0).mean() * 100
has_cents_1 = (df[df['isFraud'] == 1]['amt_cents'] > 0).mean() * 100

print(f"\n[*] Tỷ lệ giao dịch CÓ PHẦN LẺ THẬP PHÂN:")
print(f" - Nhóm HỢP LỆ (0):  {has_cents_0:.2f}%")
print(f" - Nhóm GIAN LẬN (1): {has_cents_1:.2f}%")

amt_compare = df.groupby('isFraud')['TransactionAmt'].agg(['count', 'mean', 'median', 'std', 'max'])
print("\n[*] So sánh chi tiêu giữa 2 nhóm:")
display(amt_compare)

---
### Khối 3: Phân tích 14 Cột Đếm `C1 - C14`
> **Ý nghĩa tài liệu:** Các cột $C$ đếm tần suất xuất hiện của số điện thoại, email, địa chỉ IP liên kết với thẻ thanh toán.

In [ ]:
c_cols = [f'C{i}' for i in range(1, 15)]

c_summary = pd.DataFrame({
    'Missing_Rate (%)': df[c_cols].isna().mean() * 100,
    'Min': df[c_cols].min(),
    'Median': df[c_cols].median(),
    'Mean': df[c_cols].mean(),
    'Percentile_95': df[c_cols].quantile(0.95),
    'Percentile_99': df[c_cols].quantile(0.99),
    'Max': df[c_cols].max()
})

print("[*] BẢNG PHÂN PHỐI 14 CỘT ĐẾM C1 - C14:")
display(c_summary)

print("\n[*] Nhận xét sơ bộ:")
print(" - Các cột C có giá trị trung vị (Median) rất nhỏ (1.0 hoặc 0.0), nhưng giá trị Max lên tới hàng nghìn!")
print(" - Đây là dạng phân phối lệch đuôi cực nặng (Extreme Right Skew). Khi đưa vào Deep Learning, BẮT BUỘC phải dùng phép biến đổi Log: np.log1p(x)!")

---
### Khối 4: Phân tích 15 Cột Khoảng cách Thời gian `D1 - D15`
> **Ý nghĩa tài liệu:** Các cột $D$ đo khoảng cách ngày (timedelta) giữa các sự kiện trước đó và giao dịch hiện tại.

In [ ]:
d_cols = [f'D{i}' for i in range(1, 16)]

d_summary = pd.DataFrame({
    'Missing_Rate (%)': df[d_cols].isna().mean() * 100,
    'Min': df[d_cols].min(),
    'Median': df[d_cols].median(),
    'Max': df[d_cols].max(),
    'Negative_Count': (df[d_cols] < 0).sum()
})

print("[*] BẢNG PHÂN PHỐI 15 CỘT KHOẢNG CÁCH THỜI GIAN D1 - D15:")
display(d_summary)

---
### Khối 5: Gom cụm 339 Cột `V` theo Mẫu hình Thiếu (15 Cụm Mẹ)

In [ ]:
v_cols = [f'V{i}' for i in range(1, 340) if f'V{i}' in df.columns]

# Nhóm các cột V có cùng số lượng NaN
v_nan_groups = {}
for col in v_cols:
    nan_sum = df[col].isna().sum()
    if nan_sum not in v_nan_groups:
        v_nan_groups[nan_sum] = []
    v_nan_groups[nan_sum].append(col)

print("=" * 75)
print(f"[*] KẾT QUẢ GOM CỤM 339 CỘT V THEO MẪU HÌNH THIẾU:")
print(f" - Tổng số cột V ban đầu: {len(v_cols)} cột")
print(f" - Số cụm tính năng mẹ được phát hiện: {len(v_nan_groups)} cụm")
print("=" * 75)

# Sắp xếp theo số lượng cột giảm dần
sorted_clusters = sorted(v_nan_groups.items(), key=lambda x: len(x[1]), reverse=True)

for idx, (nan_cnt, cols) in enumerate(sorted_clusters, 1):
    pct = (nan_cnt / len(df)) * 100
    # Tìm dải cột nhỏ nhất và lớn nhất
    indices = [int(c.replace('V', '')) for c in cols]
    min_idx, max_idx = min(indices), max(indices)
    print(f"Cụm {idx:02d}: {len(cols):>2} cột V (Dải V{min_idx} -> V{max_idx}) | Missing: {pct:5.2f}% ({nan_cnt:,} dòng)")

---
### Khối 6: SOI SÂU 15 CỤM MẸ CỘT V (Bản chất nội dung, Độ trùng lặp & Chọn lọc đại diện)

Trong phần này, ta giải quyết 3 câu hỏi lớn:
1. **Bản chất giá trị:** Các cột trong cụm là dạng gì? Cờ nhị phân (0/1), Biến đếm (Counts), hay Tiền tích lũy (Dollar Amounts)?
2. **Độ trùng lặp:** Tính ma trận tương quan (Pearson Correlation) trung bình giữa các cột trong từng cụm để kiểm tra mức độ đa cộng tuyến (collinearity).
3. **Lựa chọn đại diện tinh hoa:** Với mỗi cụm, tìm ra đúng **1 cột có tương quan cao nhất với nhãn `isFraud`** để dùng làm đại diện, giảm từ 339 cột xuống đúng 15 cột!

In [ ]:
cluster_analysis = []

print("[*] Đang phân tích sâu từng cụm trong 15 cụm mẹ...")

for idx, (nan_cnt, cols) in enumerate(sorted_clusters, 1):
    sub_df = df[cols]
    
    # 1. Phân loại kiểu giá trị dựa trên max và unique
    max_val = sub_df.max().max()
    min_val = sub_df.min().min()
    
    # Đánh giá bản chất dữ liệu trong cụm
    if max_val <= 1.0 and min_val >= 0.0:
        content_type = "Cờ Nhị Phân (Binary Match Flags)"
    elif max_val > 1000.0:
        content_type = "Tiền Tích Lũy / Lượng Lớn (Aggregated Amounts)"
    else:
        content_type = "Biến Đếm Tần Suất (Velocity Counts)"
        
    # 2. Tính tương quan nội cụm (Lấy mẫu 50,000 dòng để tính cực nhanh)
    sample_df = sub_df.sample(n=min(50000, len(sub_df)), random_state=42)
    corr_matrix = sample_df.corr().abs()
    
    # Lấy trung bình tương quan giữa các cặp cột khác nhau
    np.fill_diagonal(corr_matrix.values, np.nan)
    avg_corr = corr_matrix.mean().mean()
    
    # 3. Tìm cột có tương quan cao nhất với isFraud trong cụm này
    # Tính tương quan của từng cột trong cụm với nhãn mục tiêu
    fraud_corrs = df[cols].corrwith(df['isFraud']).abs()
    best_col = fraud_corrs.idxmax()
    best_corr_val = fraud_corrs.max()
    
    # Lấy dải index
    indices = [int(c.replace('V', '')) for c in cols]
    span = f"V{min(indices)}-V{max(indices)}"
    
    cluster_analysis.append({
        'Cụm': f"Cụm {idx:02d}",
        'Dải Cột': span,
        'Số Cột': len(cols),
        'Missing (%)': f"{(nan_cnt / len(df)) * 100:.1f}%",
        'Bản Chất Dữ Liệu': content_type,
        'Độ Trùng Lặp (Avg Corr)': f"{avg_corr:.2f}" if not np.isnan(avg_corr) else "N/A",
        'Cột Đại Diện Tốt Nhất': best_col,
        'Tương Quan với isFraud': f"{best_corr_val:.4f}"
    })

# Hiển thị bảng tổng phổ phân tích 15 Cụm
df_clusters = pd.DataFrame(cluster_analysis)
print("=" * 95)
print("BẢNG GIẢI MÃ BẢN CHẤT & ĐẠI DIỆN TINH HOA CỦA 15 CỤM CỘT V:")
print("=" * 95)
display(df_clusters)

# 4. Vẽ Heatmap minh họa độ trùng lặp khổng lồ của Cụm 02 (Dải V95 - V100)
sample_cluster2_cols = sorted_clusters[1][1][:12] # Lấy mẫu 12 cột đầu của Cụm 2
plt.figure(figsize=(9, 7))
sns.heatmap(df[sample_cluster2_cols].corr(), annot=True, fmt='.2f', cmap='Blues', cbar=True)
plt.title(f'Minh chứng Đa Cộng Tuyến Cực Đoan trong Cụm 02 ({sample_cluster2_cols[0]} đến {sample_cluster2_cols[-1]})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("[*] KẾT LUẬN CHIẾN LƯỢC CHO DEEP LEARNING:")
print(f"Thay vì ném cả 339 cột vào mạng nơ-ron gây nổ tham số và học vẹt, ta CHỈ CẦN LẤY:")
print(f"Danh sách 15 cột đại diện: {df_clusters['Cột Đại Diện Tốt Nhất'].tolist()}")
print("-> Giảm 95.5% chiều dữ liệu cột V mà vẫn giữ lại gần như toàn bộ thông tin dự báo gian lận!")